# Machine Learning sur du texte avec TF-IDF et LSA

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
import numpy
import pandas
import pathlib
import seaborn
import sklearn.decomposition
import sklearn.feature_extraction
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import sklearn.pipeline

## Récupération des données
Exécutez la cellule suivante pour récupérer les deux corpus de ce TP.

Le premier est une collection de résumés de films tirés de Wikipédia, avec des métadonnées telles que l'année de sortie, le titre ou le genre.

Le second, 20 Newsgroups, rassemble les messages de 20 forums de discussion : il servira à l'analyse sémantique latente.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh wikipedia-movie-plots
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
chemin_films = "wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Chargement des données

Exécutez la cellule suivante. Elle charge le fichier CSV dans un dataframe pandas, `df`, et affiche ses colonnes et sa taille : vous devez trouver 34 886 films. Une ligne est un film ; la colonne `Genre` donne son genre, et `Plot` le résumé de son intrigue, en anglais.

In [ ]:
df = pandas.read_csv(chemin_films)
print(f"Colonnes : {', '.join(df.columns)}")
print(f"Taille du dataframe : {df.shape}")

## Sélection de données pour la suite

Pour des raisons de représentativité, on ne conserve que les 5 genres les plus représentés. Le genre le plus fréquent, `unknown`, n'en fait évidemment pas partie.

Quels sont les genres conservés ? Construisez un nouveau dataframe, `df_propre`, qui ne contient que les films de ces genres, et tracez le nombre de films par genre. Combien de films reste-t-il ?

**Documentation utile**

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.drop`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.drop.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)
- [`seaborn.countplot`](https://seaborn.pydata.org/generated/seaborn.countplot.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`df["Genre"].value_counts()` compte les films de chaque genre, du plus fréquent au moins fréquent ; `.drop("unknown")` retire `unknown` de ce compte, et `.index.values[:5]` donne les 5 premiers genres qui restent. `df["Genre"].isin(genres)` vaut `True` pour les films de l'un de ces genres : il sélectionne les lignes de `df`. `seaborn.countplot(df_propre, x="Genre")` trace le nombre de films de chaque genre.

Vous devez garder `drama`, `comedy`, `horror`, `action` et `thriller`, soit 13 574 films.

```python
comptes = df["Genre"].value_counts().drop(...)
genres = comptes.index.values[:...]
print(f"Genres conservés : {', '.join(genres)}")
print(f"Nombre de films conservés : {comptes[genres].sum()}")

df_propre = df[...]
seaborn.countplot(df_propre, x="Genre")
```

### Solution

In [ ]:
comptes = df["Genre"].value_counts().drop("unknown")
genres = comptes.index.values[:5]
print(f"Genres conservés : {', '.join(genres)}")
print(f"Nombre de films conservés : {comptes[genres].sum()}")

In [ ]:
df_propre = df[df["Genre"].isin(genres)]
seaborn.countplot(df_propre, x="Genre")

## Transformation TF-IDF

Convertissez les résumés `x` en vecteurs avec `TfidfVectorizer` : supprimez les mots vides et ne gardez que les 3000 mots les plus fréquents. Appelez `vectoriseur` le vectoriseur et `X` la matrice obtenue, puis affichez la taille de `X` et les 10 premiers mots du vocabulaire.

**Documentation utile**

- [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
x = df_propre["Plot"]

# Votre code ici

### Conseils supplémentaires

`stop_words="english"` retire les mots vides anglais, la langue des résumés, et `max_features=3000` garde les 3000 mots les plus fréquents. `fit_transform(x)` apprend le vocabulaire et les poids idf, puis renvoie la matrice : une ligne par film, une colonne par mot. `get_feature_names_out()` donne les mots des colonnes, dans l'ordre alphabétique.

Vous devez obtenir une matrice de taille `(13574, 3000)`.

```python
vectoriseur = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words=..., max_features=...
)
X = vectoriseur.fit_transform(...)
noms_variables = vectoriseur.get_feature_names_out()
print(f"Taille de la matrice : {X.shape}")
print(f"10 premières colonnes : {', '.join(noms_variables[:10])}")
```

### Solution

In [ ]:
vectoriseur = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000
)
X = vectoriseur.fit_transform(x)
noms_variables = vectoriseur.get_feature_names_out()
print(f"Taille de la matrice : {X.shape}")
print(f"10 premières colonnes : {', '.join(noms_variables[:10])}")

## Poids TF-IDF d'un film

Affichez le résumé du premier film (`x.iloc[0]`) et les 5 mots qui ont le plus fort poids dans sa ligne de `X`, avec leur poids : disent-ils de quoi parle le film ?

Affichez ensuite, pour `time` et pour `clouseau` (l'inspecteur de *La Panthère rose*), le nombre de résumés qui contiennent le mot, son $df_i$, et son idf, que donne `vectoriseur.idf_`. Retrouvez ce que dit la diapositive « TF-IDF — Intérêt de $df_i$ » : lequel des deux mots pèse le plus quand il apparaît, et pourquoi ?

Comparez enfin ces idf au $\ln\frac{N}{df_i}$ de la formule du cours, où $N$ est le nombre de résumés. `TfidfVectorizer` calcule un idf lissé, $\ln\frac{1 + N}{1 + df_i} + 1$, puis divise chaque ligne de `X` par sa norme : quel écart cela fait-il ?

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`X[0]` est la ligne du premier film, une matrice creuse : `X[0].toarray()[0]` en fait un tableau de 3000 poids, un par mot de `noms_variables`. `numpy.argsort(poids_0)` trie ses indices du plus petit poids au plus grand : `[::-1][:5]` donne ceux des 5 plus forts.

`vectoriseur.vocabulary_` associe chaque mot à sa colonne, et `vectoriseur.idf_[colonne]` donne son idf. `(X[:, colonne] > 0).sum()` compte les résumés qui contiennent le mot, et `X.shape[0]` est le nombre de résumés ; `numpy.log` est le logarithme népérien.

Vous devez trouver `throwing`, `quiet` et `violent` en tête, et des idf de 2,16 pour `time` et de 8,12 pour `clouseau`.

```python
poids_0 = X[0].toarray()[0]
print(f"Résumé du premier film : {x.iloc[0]}")
for indice in numpy.argsort(poids_0)[::-1][:5]:
  print(f"{noms_variables[indice]} : {poids_0[indice]:.2f}")

nb_films = X.shape[0]
for mot in ["time", "clouseau"]:
  colonne = vectoriseur.vocabulary_[mot]
  df_i = ...
  idf_cours = ...
  print(
    f"{mot} : dans {df_i} résumés, idf {vectoriseur.idf_[colonne]:.2f}, "
    f"ln(N / df) {idf_cours:.2f}"
  )
```

### Solution

In [ ]:
poids_0 = X[0].toarray()[0]
print(f"Résumé du premier film : {x.iloc[0]}")
for indice in numpy.argsort(poids_0)[::-1][:5]:
  print(f"{noms_variables[indice]} : {poids_0[indice]:.2f}")

nb_films = X.shape[0]
for mot in ["time", "clouseau"]:
  colonne = vectoriseur.vocabulary_[mot]
  # df_i : le nombre de résumés qui contiennent le mot
  df_i = (X[:, colonne] > 0).sum()
  idf_cours = numpy.log(nb_films / df_i)
  print(
    f"{mot} : dans {df_i} résumés, idf {vectoriseur.idf_[colonne]:.2f}, "
    f"ln(N / df) {idf_cours:.2f}"
  )

# throwing, quiet et violent racontent le film : des mots assez rares (chacun dans
# 250 à 400 résumés), donc d'idf élevé, qui apparaissent dans ce résumé court.
# time est le mot le plus répandu du vocabulaire (4260 résumés sur 13574) : il a
# le plus petit idf, 2,16. clouseau, dans 10 résumés, a le plus grand, 8,12.
# La formule du cours donne 1,16 et 7,21 : le lissage ajoute environ 1 à chaque
# idf, si bien qu'un mot présent dans tous les résumés pèse encore 1 au lieu de 0.
# Les mots restent rangés dans le même ordre : les idées sont les mêmes, les
# valeurs changent, et la normalisation des lignes les change encore.

## Un corpus pour la LSA

L'analyse sémantique latente (LSA) est une SVD tronquée de la matrice documents × mots : chaque axe regroupe des mots qui apparaissent ensemble. Pour qu'elle montre quelque chose, il faut un corpus aux thèmes bien marqués. Ce n'est pas le cas des résumés de films : leurs axes se remplissent de prénoms (`joe`, `jack`, `mary`). On passe donc à 20 Newsgroups, dont les messages viennent de 20 forums thématiques : sport, informatique, religion, politique…

Exécutez la cellule suivante. Elle charge les 11 314 messages de la partie entraînement du corpus, et leur retire leur en-tête (expéditeur, sujet…), qui s'arrête à la première ligne vide.

In [ ]:
# Chaque dossier correspond à un forum, puis un message par fichier
messages = [
  p.read_text(encoding="latin-1").partition("\n\n")[2]
  for p in sorted(pathlib.Path("20news-bydate-train").glob("*/*"))
]

print(f"{len(messages)} messages récupérés")
print(messages[0][:300])

## Transformation TF-IDF des messages

Comme pour les films, convertissez `messages` en vecteurs avec `TfidfVectorizer` : supprimez les mots vides et gardez les 3000 mots les plus fréquents. Retirez aussi les mots présents dans plus de 30 % des messages, ou dans moins de 5 messages.

Affichez la taille de la matrice et le nombre moyen de valeurs non nulles par ligne : combien des 3000 mots un message emploie-t-il ?

**Documentation utile**

- [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`stop_words` et `max_features` s'utilisent comme pour les films. `max_df=0.3` retire les mots présents dans plus de 30 % des documents (un nombre à virgule est une part des documents) et `min_df=5` ceux présents dans moins de 5 documents (un entier est un nombre de documents).

`X_messages.shape` donne la taille de la matrice et `X_messages.nnz` son nombre de valeurs non nulles. Vous devez obtenir `(11314, 3000)` et environ 55 mots par message.

```python
vectoriseur_messages = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words=..., max_features=..., max_df=..., min_df=...
)
X_messages = vectoriseur_messages.fit_transform(...)
noms_variables_messages = vectoriseur_messages.get_feature_names_out()
print(f"Taille de la matrice : {X_messages.shape}")
print(f"Mots par message : {...:.0f}")
```

### Solution

In [ ]:
vectoriseur_messages = sklearn.feature_extraction.text.TfidfVectorizer(
  stop_words="english", max_features=3000, max_df=0.3, min_df=5
)
X_messages = vectoriseur_messages.fit_transform(messages)
noms_variables_messages = vectoriseur_messages.get_feature_names_out()
print(f"Taille de la matrice : {X_messages.shape}")
print(f"Mots par message : {X_messages.nnz / X_messages.shape[0]:.0f}")

## Analyse sémantique latente

Utilisez `TruncatedSVD` pour calculer une LSA à 6 axes sur `X_messages`.

Affichez la valeur singulière de chaque axe, la part de variance qu'il conserve, et la part que conservent les 6 axes ensemble. Quelle part de la variance gardent-ils ? L'axe qui a la plus grande valeur singulière garde-t-il la plus grande variance ?

**Documentation utile**

- [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`n_components=6` fixe le nombre d'axes, `n_iter=100` affine le calcul et `random_state=42` redonne les mêmes axes d'une exécution à l'autre. Après `svd.fit(X_messages)`, les attributs `singular_values_` et `explained_variance_ratio_` ont une valeur par axe.

Vous devez obtenir une valeur singulière de 16 pour l'axe 1, et environ 2,8 % de variance pour les 6 axes.

```python
svd = sklearn.decomposition.TruncatedSVD(...)
svd.fit(...)

print(f"Valeurs singulières : {svd.singular_values_.round(2)}")
print(f"Part de variance par axe : {...}")
print(f"Part de variance des 6 axes : {...:.1%}")
```

### Solution

In [ ]:
svd = sklearn.decomposition.TruncatedSVD(n_components=6, n_iter=100, random_state=42)
svd.fit(X_messages)

print(f"Valeurs singulières : {svd.singular_values_.round(2)}")
print(f"Part de variance par axe : {svd.explained_variance_ratio_.round(4)}")
print(f"Part de variance des 6 axes : {svd.explained_variance_ratio_.sum():.1%}")

## Mots importants

Les 6 axes gardent moins de 3 % de la variance : c'est normal, et ce n'est pas un échec. Un message n'emploie qu'environ 55 des 3000 mots, et l'essentiel de la variance vient de ces choix propres à chaque message : il faudrait environ 500 axes pour en garder la moitié. La LSA ne retient que ce que beaucoup de messages partagent : un axe se juge à ses mots, pas à sa part de variance.

L'axe 1 n'est pas un thème. `TruncatedSVD` ne centre pas la matrice, dont aucune valeur n'est négative : cet axe a tous ses coefficients positifs, et il donne le vocabulaire moyen du corpus. Il a ainsi la plus grande valeur singulière sans garder la plus grande variance. Les thèmes se lisent à partir de l'axe 2.

Le signe d'un coefficient porte le sens : un axe oppose le pôle de ses coefficients positifs à celui de ses coefficients négatifs. Pour chaque axe, affichez les 8 mots les plus positifs et les 8 mots les plus négatifs. Puis, pour les axes 2 à 6, nommez chaque pôle : de quel thème parle-t-il ?

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`svd.components_` a une ligne par axe et une colonne par mot. `numpy.argsort(svd.components_, axis=1)` trie chaque ligne : pour un axe, `indices` va du coefficient le plus négatif au plus positif. `indices[:8]` donne donc les 8 plus négatifs, `indices[::-1][:8]` les 8 plus positifs, et `noms_variables_messages[indices[:8]]` les mots correspondants.

Vous devez trouver `god` et `jesus` à un bout de l'axe 2, `windows` et `card` à l'autre. Seule l'opposition compte : quel pôle est positif ne dépend que du calcul.

```python
nb_mots = 8

for numero, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positifs = ...
  negatifs = ...
  print(f"Axe {numero}")
  print(f"  + {', '.join(positifs)}")
  print(f"  - {', '.join(negatifs)}")
```

### Solution

In [ ]:
nb_mots = 8

# argsort trie les coefficients de chaque axe, du plus négatif au plus positif
for numero, indices in enumerate(numpy.argsort(svd.components_, axis=1), start=1):
  positifs = noms_variables_messages[indices[::-1][:nb_mots]]
  negatifs = noms_variables_messages[indices[:nb_mots]]
  print(f"Axe {numero}")
  print(f"  + {', '.join(positifs)}")
  print(f"  - {', '.join(negatifs)}")

# Axe 1 : le vocabulaire moyen. Ses coefficients sont tous positifs : les mots
# affichés comme négatifs sont juste les plus proches de 0.
# Axe 2 : religion contre informatique
# Axe 3 : religion contre sport
# Axe 4 : cryptographie contre religion et sport
# Axe 5 : matériel contre logiciel
# Axe 6 : cryptographie contre Proche-Orient
# Les pôles ne sont pas purs (`windows` côté religion sur l'axe 3) : les axes
# doivent rester orthogonaux, si bien qu'un thème se répartit sur plusieurs axes.

## Approche supervisée

Retour aux films : on va essayer de prédire le genre d'un film à partir de son résumé.

Pour cela, commencez par extraire le genre de chaque film dans une série pandas `y`. Les classifieurs de scikit-learn prennent ces chaînes de caractères telles quelles.

`drama` compte 6 fois plus de films que `thriller`. Quelle exactitude obtient un modèle qui répond toujours `drama` ? C'est le score à battre.

**Documentation utile**

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`df_propre["Genre"]` donne la série des genres. `y.value_counts(normalize=True)` donne la part de chaque genre au lieu de son nombre de films : celle de `drama` est l'exactitude d'un modèle qui répond toujours `drama`.

Vous devez trouver 43,9 %.

```python
y = df_propre[...]

parts = y.value_counts(normalize=...)
display(parts)

print(f"Exactitude en répondant toujours drama : {parts[...]:.1%}")
```

### Solution

In [ ]:
y = df_propre["Genre"]

parts = y.value_counts(normalize=True)
display(parts)

print(f"Exactitude en répondant toujours drama : {parts['drama']:.1%}")

## Apprentissage

Apprenez une régression logistique qui prend en entrée les vecteurs TF-IDF des résumés et prédit le genre. Elle doit faire mieux que le modèle constant de la section précédente.

Évaluez-la par validation croisée à 5 plis, avec `cross_val_predict`. Le vocabulaire et les poids idf font partie du modèle : pour qu'ils ne soient appris que sur les données d'entraînement de chaque pli, mettez le `TfidfVectorizer` et le classifieur dans un même pipeline (`make_pipeline`), appliqué aux textes `x`. Le fichier est trié par origine des films : pour que les plis ne suivent pas ces blocs, mélangez-les avec `StratifiedKFold`.

Affichez l'exactitude, le macro-F1 et la matrice de confusion. Le macro-F1 est la moyenne des F1 de chaque genre : `thriller` y compte autant que `drama`, alors que l'exactitude dépend surtout des grands genres. Écrivez cette évaluation dans une fonction qui reçoit le modèle à placer après le vectoriseur : elle va servir trois fois de plus.

Comparez ensuite :

- la même régression logistique avec `class_weight="balanced"`, qui donne plus de poids aux films des petits genres : que gagne le macro-F1, que perd l'exactitude, quels genres changent dans la matrice de confusion ?
- une LSA entre le vectoriseur et la régression logistique : insérez `TruncatedSVD` dans le pipeline, à 5 puis à 100 axes. Que perd-on par rapport aux 3000 mots ?

**Documentation utile**

- [`sklearn.linear_model.LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [`sklearn.pipeline.make_pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [`sklearn.model_selection.StratifiedKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)
- [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
- [`sklearn.metrics.confusion_matrix`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)
- [`seaborn.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`make_pipeline(vectoriseur, modele)` enchaîne les étapes. Un pipeline est lui-même un modèle : `make_pipeline(TruncatedSVD(...), LogisticRegression(...))` se passe à la fonction comme `modele`, pour insérer la LSA. `LogisticRegression(max_iter=200)` lui laisse assez d'itérations pour converger, et `TruncatedSVD(5, random_state=0)` garde 5 axes.

`StratifiedKFold(5, shuffle=True, random_state=0)` mélange les films et garde la part de chaque genre dans chaque pli ; `cross_val_predict(modele, x, y, cv=plis)` renvoie, pour chaque film, la prédiction d'un modèle appris sans lui. `f1_score(y, predictions, average="macro")` donne le macro-F1. `confusion_matrix(y, predictions, normalize="true")` donne, pour chaque genre réel (une ligne), la part de ses films prédite dans chaque genre (une colonne), genres rangés dans l'ordre alphabétique (`sorted(genres)`) ; `seaborn.heatmap` l'affiche.

Vous devez obtenir une exactitude de 0,66 et un macro-F1 de 0,55 avec la régression logistique seule.

```python
def evalue_pipeline(modele):
  modele = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(
      stop_words="english", max_features=3000
    ),
    modele,
  )
  plis = sklearn.model_selection.StratifiedKFold(...)
  predictions = sklearn.model_selection.cross_val_predict(...)

  exactitude = sklearn.metrics.accuracy_score(y, predictions)
  macro_f1 = sklearn.metrics.f1_score(...)
  print(f"Exactitude : {exactitude:.2f}, macro-F1 : {macro_f1:.2f}")

  matrice_confusion = sklearn.metrics.confusion_matrix(...)
  seaborn.heatmap(
    matrice_confusion,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    xticklabels=sorted(genres),
    yticklabels=sorted(genres),
    cmap="Blues",
  )
  plt.xlabel("Genre prédit")
  plt.ylabel("Genre réel")
  plt.show()


evalue_pipeline(sklearn.linear_model.LogisticRegression(max_iter=200))
evalue_pipeline(sklearn.linear_model.LogisticRegression(class_weight=..., max_iter=200))
evalue_pipeline(
  sklearn.pipeline.make_pipeline(
    sklearn.decomposition.TruncatedSVD(..., random_state=0),
    sklearn.linear_model.LogisticRegression(max_iter=200),
  )
)
```

### Solution

In [ ]:
def evalue_pipeline(modele):
  # Le vectoriseur fait partie du pipeline : à chaque pli de la validation
  # croisée, il n'apprend son vocabulaire et ses poids idf que sur les données
  # d'entraînement du pli
  modele = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.TfidfVectorizer(
      stop_words="english", max_features=3000
    ),
    modele,
  )
  # Le fichier est trié par origine : on mélange pour que les plis ne suivent pas
  # ces blocs
  plis = sklearn.model_selection.StratifiedKFold(5, shuffle=True, random_state=0)
  predictions = sklearn.model_selection.cross_val_predict(modele, x, y, cv=plis)

  # L'exactitude est la part de bonnes prédictions : les grands genres y pèsent
  # le plus
  exactitude = sklearn.metrics.accuracy_score(y, predictions)
  # Le macro-F1 est la moyenne non pondérée des F1 de chaque genre : les petits
  # genres y comptent autant que les grands
  macro_f1 = sklearn.metrics.f1_score(y, predictions, average="macro")
  print(f"Exactitude : {exactitude:.2f}, macro-F1 : {macro_f1:.2f}")

  # Une ligne par genre réel, une colonne par genre prédit : chaque case donne la
  # part des films du genre réel prédits dans ce genre
  matrice_confusion = sklearn.metrics.confusion_matrix(y, predictions, normalize="true")
  seaborn.heatmap(
    matrice_confusion,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    xticklabels=sorted(genres),
    yticklabels=sorted(genres),
    cmap="Blues",
  )
  plt.xlabel("Genre prédit")
  plt.ylabel("Genre réel")
  plt.show()

In [ ]:
# Régression logistique seule
evalue_pipeline(sklearn.linear_model.LogisticRegression(max_iter=200))

# Exactitude 0,66 : mieux que les 0,44 du modèle constant. Mais macro-F1 0,55 :
# les petits genres sont mal reconnus, thriller surtout, dont 11 % des films
# seulement sont prédits thriller

In [ ]:
# Rééquilibrage : chaque genre pèse autant dans l'apprentissage
evalue_pipeline(
  sklearn.linear_model.LogisticRegression(class_weight="balanced", max_iter=200)
)

# Exactitude 0,63 (-0,03), macro-F1 0,58 (+0,03) : le modèle prédit plus souvent
# les petits genres. Il retrouve 42 % des thrillers au lieu de 11 %, et 66 % des
# films d'action au lieu de 38 %, au prix des drames (60 % au lieu de 80 %).
# Lequel garder dépend de l'usage : l'exactitude si seul compte le nombre de films
# bien classés, le macro-F1 si chaque genre compte

In [ ]:
# LSA à 5 axes entre le vectoriseur et la régression logistique
evalue_pipeline(
  sklearn.pipeline.make_pipeline(
    sklearn.decomposition.TruncatedSVD(5, random_state=0),
    sklearn.linear_model.LogisticRegression(max_iter=200),
  )
)

# Exactitude 0,46, macro-F1 0,23 : à peine mieux que le modèle constant, et
# thriller n'est plus jamais prédit. 5 axes ne gardent que le vocabulaire que
# beaucoup de résumés partagent, pas ce qui distingue les genres

In [ ]:
# LSA à 100 axes
evalue_pipeline(
  sklearn.pipeline.make_pipeline(
    sklearn.decomposition.TruncatedSVD(100, random_state=0),
    sklearn.linear_model.LogisticRegression(max_iter=200),
  )
)

# Exactitude 0,60, macro-F1 0,49 : mieux, mais toujours en dessous des 3000 mots.
# La LSA garde les mots qui apparaissent ensemble dans beaucoup de résumés, et perd
# les mots plus rares qui portent une partie du genre. Elle sert à explorer un
# corpus et à réduire sa représentation, pas ici à mieux classer

## Pour aller plus loin

TF-IDF et LSA restent de bonnes références de base, rapides et faciles à interpréter. Pour trouver les thèmes d'un corpus, on part plutôt aujourd'hui (2026) d'embeddings de phrases, que l'on regroupe par proximité : c'est ce que fait [BERTopic](https://maartengr.github.io/BERTopic/), qui décrit ensuite chaque groupe par ses mots les plus caractéristiques, avec une variante de TF-IDF.